# NYC Taxi Trip Behavior Clustering Analysis

This notebook demonstrates a simplified workflow of the project:
- Data loading and preprocessing
- Feature engineering
- Clustering with K-Means
- Visualization of clustering results

The goal is to identify interpretable travel behavior patterns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

## Data Loading & Preprocessing

In [ ]:
# 读取数据（路径根据你实际情况改）
df = pd.read_csv("../data_sample/taxi_sample.csv")

# 简单清洗（模拟你的逻辑）
df = df[
    (df["trip_distance"] > 0) &
    (df["fare_amount"] > 0) &
    (df["total_amount"] > 0)
]

df.head()

## Feature Engineering

In [ ]:
# 时间特征
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])
df["hour"] = df["pickup_datetime"].dt.hour
df["is_peak"] = df["hour"].apply(lambda x: 1 if 17 <= x <= 19 else 0)

# 空间特征（区域频率）
freq = df["pu_Location_id"].value_counts(normalize=True)
df["pu_freq"] = df["pu_Location_id"].map(freq)

# 费用结构特征
df["extra_ratio"] = (df["total_amount"] - df["fare_amount"]) / df["total_amount"]

df[["is_peak", "pu_freq", "extra_ratio"]].head()

## K-Means Clustering

In [ ]:
features = ["is_peak", "pu_freq", "total_amount", "extra_ratio", "fare_amount"]
X = df[features]

# 标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# KMeans
kmeans = KMeans(n_clusters=5, random_state=42)
labels = kmeans.fit_predict(X_scaled)

df["cluster"] = labels

df["cluster"].value_counts()

## Visualization (PCA Projection)

In [ ]:
# PCA降维
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# 绘图
plt.figure(figsize=(6, 5))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels)
plt.title("K-Means Clustering (PCA Projection)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(scatter)
plt.show()

## Key Observations

- The clustering results show clear separation in feature space
- High-frequency zones tend to form dense clusters
- Cost-related features contribute significantly to cluster separation

These results indicate that feature engineering plays a crucial role in identifying interpretable travel patterns.